# Mongolian Forced Aligner with Tacotron and aeneas

Given an audio file containing speech, and the corresponding transcript, computing a forced alignment is the process of determining, for each fragment of the transcript, the time interval (in the audio file) containing the spoken text of the fragment.

Typical applications of forced alignment include closed captioning and automating the creation of training data for **automated speech recognition** and **text-to-speech** systems.

For more information about forced alignment tools, see [pettarin/forced-alignment-tools](https://github.com/pettarin/forced-alignment-tools)

Currently, there is no Mongolian forced aligner tool. This is the first attempt to implement a forced aligner for the Mongolian language using [Rayhane-mamah/Tacotron-2](https://github.com/Rayhane-mamah/Tacotron-2) and [readbeyond/aeneas](https://github.com/readbeyond/aeneas).

For implementation details, visit [tugstugi/mongolian-nlp/forced_aligner](https://github.com/tugstugi/mongolian-nlp/tree/master/forced_aligner)

## Setup

In [1]:
import os, pathlib, json
from os.path import exists, join, expanduser

import IPython
from IPython.display import YouTubeVideo, Audio, clear_output, display

### Install aeneas

In [2]:
# aeneas needs espeak
!apt-get install -qq libespeak-dev > /dev/null
# aeneas must be installed from the devel branch
!pip install -q https://codeload.github.com/readbeyond/aeneas/zip/devel

     / 32.0 MB 24.1 MB/s 0:00:01
  Preparing metadata (setup.py) ... done


### Install Tacotron-2

In [5]:
# pyaudio needs this system dependency!
!apt-get install -qq portaudio19-dev > /dev/null
# clone Tacotron-2 and install dependencies
if not exists('Tacotron-2'):
  !git clone https://github.com/tugstugi/Tacotron-2.git && cd Tacotron-2 && pip install -q -r requirements.txt

### Download a pretrained Tacotron-2 model

In [7]:
if not exists('Tacotron-2/logs-Tacotron/taco_pretrained'):
  # download pretrained model from the Google Drive link
  pretrained_file_id = "1fgx0kpf0Oe2Idz3lUZM6-p343nc-24Hq"
  pretrained_file_name = "taco_pretrained.tar.gz"
  !curl -c ./cookie -s -L "https://drive.google.com/uc?export=download&id=$pretrained_file_id" > /dev/null
  confirm_text = !awk '/download/ {print $NF}' ./cookie
  confirm_text = confirm_text[0]
  !curl -Lb ./cookie "https://drive.google.com/uc?export=download&confirm=$confirm_text&id=$pretrained_file_id" -o $pretrained_file_name
  # extract it
  !mkdir Tacotron-2/logs-Tacotron/
  !tar xvfz $pretrained_file_name --directory Tacotron-2/logs-Tacotron/

In [6]:
!pip install -q gdown
!gdown --id 1fgx0kpf0Oe2Idz3lUZM6-p343nc-24Hq -O taco_pretrained.tar.gz
!mkdir -p Tacotron-2/logs-Tacotron/
!tar -xvzf taco_pretrained.tar.gz -C Tacotron-2/logs-Tacotron/


/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1fgx0kpf0Oe2Idz3lUZM6-p343nc-24Hq
From (redirected): https://drive.google.com/uc?id=1fgx0kpf0Oe2Idz3lUZM6-p343nc-24Hq&confirm=t&uuid=5eb97b04-8c79-4716-a507-cfd770e8e290
To: /content/taco_pretrained.tar.gz
100% 323M/323M [00:05<00:00, 59.5MB/s]
taco_pretrained/
taco_pretrained/checkpoint
taco_pretrained/tacotron_model.ckpt-150000.data-00000-of-00001
taco_pretrained/tacotron_model.ckpt-150000.meta
taco_pretrained/tacotron_model.ckpt-150000.index


### Download the Tacotron-2 wrapper for aeneas

In [8]:
if not exists('aeneas-helper.py'):
  !pip install -q youtube-dl pydub
  !curl https://raw.githubusercontent.com/tugstugi/mongolian-nlp/master/forced_aligner/aeneas-helper.py > aeneas-helper.py
  !curl https://raw.githubusercontent.com/tugstugi/mongolian-nlp/master/forced_aligner/aeneas-helper.sh > aeneas-helper.sh
  !chmod a+x aeneas-helper.sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.3 MB/s eta 0:00:00
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1237  100  1237    0     0   5483      0 --:--:-- --:--:-- --:--:--  5497
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   470  100   470    0     0   2040      0 --:--:-- --:--:-- --:--:--  2043


## Forced Aligner Demo

As a demo, we will force align a speech of the Mongolian president Battulga:

We will download the above video and extract the audio from 00:10s to 00:29s into a file **battulga.mp3**:

In [11]:
import pydub

audio = pydub.AudioSegment.from_mp3('/content/battulga_clip.mp3')
# show the audio
audio

The text of the above speech is copied from [https://president.mn/7314/](https://president.mn/7314/) and written into a file **battulga.txt**:

In [12]:
SENTENCES = [
  "Монгол Улсын дөрвөн зүг, найман зовхист амьдран суугаа хүндэт ард иргэд ээ.",
  "Даян дэлхийд тархан суурьшсан Монгол туургатнууд аа.",
  "Он солигдох торгон мөч ирлээ.",
  "Монголчууд бид гурав дахь мянганы арван наймдах оноо үдэж,",
  "ирээдүйгээ дархлан бүтээх хоёр мянга арван есөн оныг угтаж байна."
]
SENTENCES = "\n".join(SENTENCES)
!echo "$SENTENCES" > battulga.txt

Now, we will force align the audio **battulga.mp3** with the text file **battulga.txt** and write out the result into a file **result.json**. If you are using Colab and it takes too long, change your Runtime type to **GPU**.

In [17]:
!python -m aeneas.tools.execute_task \
  /content/battulga_clip.mp3 \
  battulga.txt \
  "task_language=mon|is_text_type=plain|os_task_file_format=json|tts=mon|tts_path=/content/helper.py" \
  result.json

[WARN] The default input encoding is not UTF-8.
[WARN] You might want to set 'PYTHONIOENCODING=UTF-8' in your shell.
[WARN] The default output encoding is not UTF-8.
[WARN] You might want to set 'PYTHONIOENCODING=UTF-8' in your shell.
[INFO] Validating config string (specify --skip-validator to bypass)...
[INFO] Validating config string... done
[INFO] Creating task...
[INFO] Creating task... done
[INFO] Executing task...
[ERRO] An unexpected error occurred while executing the task:
[ERRO] Unexpected error while executing task : Language 'mon' is not supported by the selected TTS engine


Read now the **result.json** file and show it nicely:

In [15]:
for fragment in json.loads(pathlib.Path('result.json').read_text())['fragments']:
    start = int(float(fragment['begin'])*1000)
    end = int(float(fragment['end'])*1000)
    text = fragment['lines'][0]
    print(text)
    display(audio[start:end])

Монгол Улсын дөрвөн зүг, найман зовхист амьдран суугаа хүндэт ард иргэд ээ.


Даян дэлхийд тархан суурьшсан Монгол туургатнууд аа.


Он солигдох торгон мөч ирлээ.


Монголчууд бид гурав дахь мянганы арван наймдах оноо үдэж,


ирээдүйгээ дархлан бүтээх хоёр мянга арван есөн оныг угтаж байна.


In [ ]:
!git clone https://github.com/readbeyond/aeneas.git
